# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [35]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

Columns:
01. content_id
02. client_id
03. search_volume
04. competition
05. competition_level
06. cpc
07. content_type
08. main_intent
09. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


In [7]:
print("\nData types:")
display(df.dtypes)


Data types:


,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


In [8]:
print("Days since last update:")
print(df["days_since_last_update"].describe())

print("\nFreshness tiers:")
display(
    df["freshness_tier"]
    .value_counts(dropna=False)
    .to_frame("n")
)

Days since last update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Freshness tiers:


,n
freshness_tier,
0-30,20480
91-180,9171
31-90,175
181+,174


In [9]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 60, 90, 180, float("inf")],
    labels=["0-30", "31-60", "61-90", "91-180", "180+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("days_since_last_update", "size"),
          avg_ctr=("ctr", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

display(staleness_table)

,staleness_bucket,n,avg_ctr,avg_position
0,0-30,20480,0.609021,15.685166
1,31-60,128,0.127266,18.384375
2,61-90,47,0.091064,11.510638
3,91-180,9171,0.238367,17.901461
4,180+,174,3.693276,11.325862


In [10]:
staleness_outcome = (
    pd.crosstab(
        df["staleness_bucket"],
        df["trend_direction"],
        normalize="index"
    ) * 100
)

display(staleness_outcome.round(2))

trend_direction,down,flat,new,stable,up
staleness_bucket,,,,,
0-30,51.14,4.38,10.39,18.62,15.47
31-60,58.59,0.00,0.00,17.19,24.22
61-90,59.57,2.13,14.89,8.51,14.89
91-180,61.11,2.58,0.83,22.89,12.59
180+,47.13,9.20,14.37,13.79,15.52


In [11]:
staleness_counts = pd.crosstab(
    df["staleness_bucket"],
    df["trend_direction"]
)

display(staleness_counts)

trend_direction,down,flat,new,stable,up
staleness_bucket,,,,,
0-30,10473,898,2128,3813,3168
31-60,75,0,0,22,31
61-90,28,1,7,4,7
91-180,5604,237,76,2099,1155
180+,82,16,25,24,27


In [12]:
print("CTR:")
display(df["ctr"].describe())

print("\nAverage position:")
display(df["avg_position"].describe())

CTR:


,ctr
count,30000.000000
mean,0.510733
std,3.279162
min,0.000000
25%,0.000000
50%,0.070000
75%,0.290000
max,100.000000



Average position:


,avg_position
count,30000.00000
mean,16.34238
std,15.21679
min,0.00000
25%,6.20000
50%,10.80000
75%,22.30000
max,245.00000


In [13]:
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-0.01, 0, 3, 5, 10, 20, float("inf")],
    labels=[
        "no_data",
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "20+"
    ]
)

position_table = (
    df.groupby("position_bucket", observed=True)
      .agg(
          n=("avg_position", "size"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

display(position_table)

,position_bucket,n,avg_ctr
0,no_data,1205,0.297369
1,1-3,1141,2.714303
2,4-5,2782,1.104820
3,6-10,9060,0.511708
4,11-20,7273,0.323443
5,20+,8539,0.211333


In [14]:
position_outcome = (
    pd.crosstab(
        df["position_bucket"],
        df["trend_direction"],
        normalize="index"
    ) * 100
)

display(position_outcome.round(2))

trend_direction,down,flat,new,stable,up
position_bucket,,,,,
no_data,0.66,3.57,95.77,0.00,0.00
1-3,49.78,14.29,14.81,15.16,5.96
4-5,55.21,6.54,5.18,22.90,10.17
6-10,57.47,5.72,2.60,20.63,13.58
11-20,60.95,1.64,2.69,21.44,13.28
20+,52.82,1.49,3.95,20.19,21.56


In [15]:
ctr_position_table = (
    df[df["avg_position"] > 0]
    .groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        p25_ctr=("ctr", lambda x: x.quantile(0.25)),
        p75_ctr=("ctr", lambda x: x.quantile(0.75))
    )
    .reset_index()
)

display(ctr_position_table)

,position_bucket,n,median_ctr,mean_ctr,p25_ctr,p75_ctr
0,1-3,1141,0.00,2.714303,0.0,0.36
1,4-5,2782,0.23,1.104820,0.0,0.56
2,6-10,9060,0.14,0.511708,0.0,0.37
3,11-20,7273,0.10,0.323443,0.0,0.30
4,20+,8539,0.00,0.211333,0.0,0.15


In [16]:
# Share of declining items by staleness bucket
staleness_summary = (
    df.assign(
        is_declining=df["trend_direction"].eq("declining")
    )
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("days_since_last_update", "size"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

staleness_summary["declining_rate_pct"] = (
    staleness_summary["declining_rate"] * 100
)

display(
    staleness_summary[
        ["staleness_bucket", "n", "declining_rate_pct"]
    ].round(2)
)

,staleness_bucket,n,declining_rate_pct
0,0-30,20480,0.0
1,31-60,128,0.0
2,61-90,47,0.0
3,91-180,9171,0.0
4,180+,174,0.0


In [17]:
ctr_position_summary = (
    df[df["avg_position"] > 0]
    .groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

display(ctr_position_summary.round(2))

,position_bucket,n,median_ctr,mean_ctr
0,1-3,1141,0.00,2.71
1,4-5,2782,0.23,1.10
2,6-10,9060,0.14,0.51
3,11-20,7273,0.10,0.32
4,20+,8539,0.00,0.21


In [18]:
print("trend_direction values:")
display(
    df["trend_direction"]
    .value_counts(dropna=False)
    .to_frame("n")
)

trend_direction values:


,n
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [19]:
print("Unique trend_direction values:")
print(df["trend_direction"].unique())

Unique trend_direction values:
['down' 'stable' 'new' 'up' 'flat']


In [20]:
staleness_counts = pd.crosstab(
    df["staleness_bucket"],
    df["trend_direction"]
)

display(staleness_counts)

trend_direction,down,flat,new,stable,up
staleness_bucket,,,,,
0-30,10473,898,2128,3813,3168
31-60,75,0,0,22,31
61-90,28,1,7,4,7
91-180,5604,237,76,2099,1155
180+,82,16,25,24,27


In [21]:
staleness_pct = pd.crosstab(
    df["staleness_bucket"],
    df["trend_direction"],
    normalize="index"
) * 100

display(staleness_pct.round(2))

trend_direction,down,flat,new,stable,up
staleness_bucket,,,,,
0-30,51.14,4.38,10.39,18.62,15.47
31-60,58.59,0.00,0.00,17.19,24.22
61-90,59.57,2.13,14.89,8.51,14.89
91-180,61.11,2.58,0.83,22.89,12.59
180+,47.13,9.20,14.37,13.79,15.52


In [22]:
valid_position = df["avg_position"] > 0

df.loc[valid_position, "ctr_position_rank"] = (
    df.loc[valid_position]
      .groupby("position_bucket")["ctr"]
      .rank(pct=True)
)

df["ctr_position_signal"] = pd.cut(
    df["ctr_position_rank"],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["lowest_25", "low_25_50", "high_50_75", "highest_25"],
    include_lowest=True
)

ctr_position_audit = (
    df[valid_position]
    .groupby(
        ["position_bucket", "ctr_position_signal"],
        observed=True
    )
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

display(ctr_position_audit)

/tmp/ipykernel_788/1743692803.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")["ctr"]


,position_bucket,ctr_position_signal,n,median_ctr,mean_ctr
0,1-3,low_25_50,588,0.00,0.000000
1,1-3,high_50_75,268,0.15,0.164590
2,1-3,highest_25,285,0.97,10.711965
3,4-5,lowest_25,858,0.00,0.000000
4,4-5,low_25_50,542,0.15,0.144354
5,4-5,high_50_75,682,0.36,0.372845
6,4-5,highest_25,700,0.94,3.915843
7,6-10,lowest_25,3188,0.00,0.000000
8,6-10,low_25_50,1288,0.09,0.083199
9,6-10,high_50_75,2351,0.23,0.237558


In [23]:
display(df["trend_direction"].value_counts(dropna=False).to_frame("n"))

,n
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [24]:
display(staleness_pct.round(2))

trend_direction,down,flat,new,stable,up
staleness_bucket,,,,,
0-30,51.14,4.38,10.39,18.62,15.47
31-60,58.59,0.00,0.00,17.19,24.22
61-90,59.57,2.13,14.89,8.51,14.89
91-180,61.11,2.58,0.83,22.89,12.59
180+,47.13,9.20,14.37,13.79,15.52


In [25]:
display(ctr_position_audit)

,position_bucket,ctr_position_signal,n,median_ctr,mean_ctr
0,1-3,low_25_50,588,0.00,0.000000
1,1-3,high_50_75,268,0.15,0.164590
2,1-3,highest_25,285,0.97,10.711965
3,4-5,lowest_25,858,0.00,0.000000
4,4-5,low_25_50,542,0.15,0.144354
5,4-5,high_50_75,682,0.36,0.372845
6,4-5,highest_25,700,0.94,3.915843
7,6-10,lowest_25,3188,0.00,0.000000
8,6-10,low_25_50,1288,0.09,0.083199
9,6-10,high_50_75,2351,0.23,0.237558


In [26]:
ctr_signal_outcome = (
    pd.crosstab(
        [
            df["position_bucket"],
            df["ctr_position_signal"]
        ],
        df["trend_direction"],
        normalize="index"
    ) * 100
)

display(ctr_signal_outcome.round(2))

trend_direction                       down   flat    new  stable     up
position_bucket ctr_position_signal                                    
1-3             low_25_50            37.07  25.85  23.13    9.69   4.25
                high_50_75           87.31   0.00   0.00    7.09   5.60
                highest_25           40.70   3.86  11.58   34.04   9.82
4-5             lowest_25            49.77  20.16  13.17    8.39   8.51
                low_25_50            74.17   0.18   0.00   17.34   8.30
                high_50_75           57.48   0.00   0.00   33.87   8.65
                highest_25           45.00   1.14   4.43   34.29  15.14
6-10            lowest_25            60.54  15.43   5.27    8.97   9.79
                low_25_50            62.66   0.00   0.93   25.00  11.41
                high_50_75           56.66   0.00   0.81   27.48  15.06
                highest_25           50.96   1.16   1.66   27.54  18.67
11-20           lowest_25            63.98   4.03   4.45   15.66  11.88
                low_25_50            66.07   0.00   1.15   22.83   9.95
                high_50_75           59.53   0.00   1.74   25.24  13.49
                highest_25           55.50   0.22   1.56   26.08  16.65
20+             low_25_50            47.57   2.63   6.52   17.06  26.23
                high_50_75           67.18   0.00   0.05   22.15  10.62
                highest_25           51.33   0.37   1.92   25.11  21.27

In [27]:
ctr_signal_counts = pd.crosstab(
    [
        df["position_bucket"],
        df["ctr_position_signal"]
    ],
    df["trend_direction"]
)

display(ctr_signal_counts)

trend_direction                      down  flat  new  stable    up
position_bucket ctr_position_signal                               
1-3             low_25_50             218   152  136      57    25
                high_50_75            234     0    0      19    15
                highest_25            116    11   33      97    28
4-5             lowest_25             427   173  113      72    73
                low_25_50             402     1    0      94    45
                high_50_75            392     0    0     231    59
                highest_25            315     8   31     240   106
6-10            lowest_25            1930   492  168     286   312
                low_25_50             807     0   12     322   147
                high_50_75           1332     0   19     646   354
                highest_25           1138    26   37     615   417
11-20           lowest_25            1826   115  127     447   339
                low_25_50             518     0    9     179    78
                high_50_75           1059     0   31     449   240
                highest_25           1030     4   29     484   309
20+             low_25_50            2153   119  295     772  1187
                high_50_75           1259     0    1     415   199
                highest_25           1098     8   41     537   455

In [28]:
ctr_down_summary = (
    df[df["avg_position"] > 0]
    .assign(is_down=df["trend_direction"].eq("down"))
    .groupby(
        ["position_bucket", "ctr_position_signal"],
        observed=True
    )
    .agg(
        n=("is_down", "size"),
        down_rate=("is_down", "mean")
    )
    .reset_index()
)

ctr_down_summary["down_rate_pct"] = (
    ctr_down_summary["down_rate"] * 100
)

display(
    ctr_down_summary[
        [
            "position_bucket",
            "ctr_position_signal",
            "n",
            "down_rate_pct"
        ]
    ].round(2)
)

,position_bucket,ctr_position_signal,n,down_rate_pct
0,1-3,low_25_50,588,37.07
1,1-3,high_50_75,268,87.31
2,1-3,highest_25,285,40.70
3,4-5,lowest_25,858,49.77
4,4-5,low_25_50,542,74.17
5,4-5,high_50_75,682,57.48
6,4-5,highest_25,700,45.00
7,6-10,lowest_25,3188,60.54
8,6-10,low_25_50,1288,62.66
9,6-10,high_50_75,2351,56.66


In [29]:
# Create search-volume quartile buckets robustly
volume_rank = df["search_volume"].rank(method="first")

df["volume_bucket"] = pd.qcut(
    volume_rank,
    q=4,
    labels=["lowest_25", "low_25_50", "high_50_75", "highest_25"]
)

volume_table = (
    df.groupby("volume_bucket", observed=True)
      .agg(
          n=("search_volume", "size"),
          median_volume=("search_volume", "median"),
          mean_volume=("search_volume", "mean")
      )
      .reset_index()
)

display(volume_table)

,volume_bucket,n,median_volume,mean_volume
0,lowest_25,6883,0.0,0.000000
1,low_25_50,6883,0.0,3.900915
2,high_50_75,6883,10.0,13.279093
3,highest_25,6883,90.0,618.349557


In [30]:
volume_down_summary = (
    df.assign(is_down=df["trend_direction"].eq("down"))
      .groupby("volume_bucket", observed=True)
      .agg(
          n=("is_down", "size"),
          down_rate=("is_down", "mean")
      )
      .reset_index()
)

volume_down_summary["down_rate_pct"] = (
    volume_down_summary["down_rate"] * 100
)

display(
    volume_down_summary[
        ["volume_bucket", "n", "down_rate_pct"]
    ].round(2)
)

,volume_bucket,n,down_rate_pct
0,lowest_25,6883,62.82
1,low_25_50,6883,59.80
2,high_50_75,6883,51.98
3,highest_25,6883,50.95


In [31]:
impression_rank = df["impressions_90d"].rank(method="first")

df["impression_bucket"] = pd.qcut(
    impression_rank,
    q=4,
    labels=["lowest_25", "low_25_50", "high_50_75", "highest_25"]
)

impression_table = (
    df.groupby("impression_bucket", observed=True)
      .agg(
          n=("impressions_90d", "size"),
          median_impressions=("impressions_90d", "median"),
          mean_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

display(impression_table)

,impression_bucket,n,median_impressions,mean_impressions
0,lowest_25,7500,10.0,19.644000
1,low_25_50,7500,300.0,334.620133
2,high_50_75,7500,1615.5,1784.976933
3,highest_25,7500,9579.5,18662.224133


In [33]:
impression_down_summary = (
    df.assign(is_down=df["trend_direction"].eq("down"))
      .groupby("impression_bucket", observed=True)
      .agg(
          n=("is_down", "size"),
          down_rate=("is_down", "mean")
      )
      .reset_index()
)

impression_down_summary["down_rate_pct"] = (
    impression_down_summary["down_rate"] * 100
)

display(
    impression_down_summary[
        ["impression_bucket", "n", "down_rate_pct"]
    ].round(2)
)

,impression_bucket,n,down_rate_pct
0,lowest_25,7500,37.60
1,low_25_50,7500,60.47
2,high_50_75,7500,62.56
3,highest_25,7500,56.20


In [34]:
visible_df = df[
    (df["impressions_90d"] > 0) &
    (df["avg_position"] > 0)
].copy()

visible_df["ctr_position_rank"] = (
    visible_df
    .groupby("position_bucket")["ctr"]
    .rank(pct=True)
)

visible_df["ctr_opportunity"] = (
    visible_df["ctr_position_rank"] <= 0.25
).astype(int)

print("Visible rows:", len(visible_df))

display(
    visible_df.groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        opportunity_count=("ctr_opportunity", "sum"),
        opportunity_rate=("ctr_opportunity", "mean")
    )
    .reset_index()
)

Visible rows: 28795


/tmp/ipykernel_788/2837068897.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")["ctr"]


,position_bucket,n,opportunity_count,opportunity_rate
0,1-3,1141,0,0.000000
1,4-5,2782,858,0.308411
2,6-10,9060,3188,0.351876
3,11-20,7273,2854,0.392410
4,20+,8539,0,0.000000


### Rule

Prioritize pages that have meaningful search visibility and unusually low
CTR compared with other pages at a similar average position. Pages without
a valid average position are not ranked by this rule.

### Signal verdicts

1. CTR relative to position — CONFIRMED
   Within position buckets, a meaningful share of pages fall into the
   lowest CTR quartile, especially at positions 4–20. This supports using
   low CTR relative to position as the main opportunity signal.

2. Impressions / visibility — MIXED
   Higher-impression groups have higher down rates than the lowest-volume
   group, but the relationship is not monotonic. Therefore impressions
   will be used only as a transparent visibility gate, not as a learned
   positive weight.

### Reason code

CTR_FIX_OPPORTUNITY

### Action

CTR_REVIEW

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [36]:
queue = df.copy()

# avg_position == 0 means no position data
queue["valid_position"] = queue["avg_position"] > 0

print("Total rows:", len(queue))
print("Rows with valid position:", queue["valid_position"].sum())

Total rows: 30000
Rows with valid position: 28795


In [37]:
queue["ctr_position_rank"] = np.nan

valid = queue["avg_position"] > 0

queue.loc[valid, "ctr_position_rank"] = (
    queue.loc[valid]
    .groupby("position_bucket")["ctr"]
    .rank(pct=True)
)

display(
    queue.loc[valid, [
        "avg_position",
        "ctr",
        "position_bucket",
        "ctr_position_rank"
    ]].head(10)
)

/tmp/ipykernel_788/3383383132.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("position_bucket")["ctr"]


,avg_position,ctr,position_bucket,ctr_position_rank
0,10.6,0.76,11-20,0.924447
1,20.3,0.05,20+,0.593571
2,36.5,0.09,20+,0.663485
3,6.2,0.49,6-10,0.818377
4,44.0,0.13,20+,0.730413
5,8.5,0.03,6-10,0.361313
6,7.0,0.00,6-10,0.175993
7,21.2,0.06,20+,0.611371
8,46.0,0.09,20+,0.663485
9,4.9,0.16,4-5,0.417326


In [38]:
visible_population = queue.loc[
    queue["valid_position"] & (queue["impressions_90d"] > 0)
]

impression_threshold = visible_population["impressions_90d"].median()

print("Impression threshold:", impression_threshold)

Impression threshold: 828.0


In [46]:
# Transparent baseline score:
# low CTR relative to position + meaningful visibility

queue["score"] = 0.0

eligible = (
    (queue["avg_position"] > 0) &
    (queue["impressions_90d"] >= impression_threshold)
)

low_ctr = queue["ctr_position_rank"] <= 0.25

queue.loc[eligible & low_ctr, "score"] = (
    (1 - queue.loc[eligible & low_ctr, "ctr_position_rank"])
    * np.log1p(queue.loc[eligible & low_ctr, "impressions_90d"])
)

In [47]:
queue["reason_code"] = np.where(
    queue["score"] > 0,
    "CTR_FIX_OPPORTUNITY",
    "NO_OPPORTUNITY"
)

In [48]:
queue["action"] = np.where(
    queue["score"] > 0,
    "CTR_REVIEW",
    "NO_ACTION"
)

In [49]:
queue = queue.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "score",
            "action",
            "reason_code",
            "impressions_90d",
            "avg_position",
            "ctr",
            "ctr_position_rank"
        ]
    ].head(20)
)

,rank,score,action,reason_code,impressions_90d,avg_position,ctr,ctr_position_rank
0,1,10.092888,CTR_REVIEW,CTR_FIX_OPPORTUNITY,208678,9.7,0.0,0.175993
1,2,8.256017,CTR_REVIEW,CTR_FIX_OPPORTUNITY,22456,6.6,0.0,0.175993
2,3,8.053187,CTR_REVIEW,CTR_FIX_OPPORTUNITY,13676,4.3,0.0,0.154385
3,4,8.016233,CTR_REVIEW,CTR_FIX_OPPORTUNITY,16786,5.6,0.0,0.175993
4,5,7.984714,CTR_REVIEW,CTR_FIX_OPPORTUNITY,16156,9.0,0.0,0.175993
5,6,7.929072,CTR_REVIEW,CTR_FIX_OPPORTUNITY,15101,5.7,0.0,0.175993
6,7,7.896688,CTR_REVIEW,CTR_FIX_OPPORTUNITY,14519,7.4,0.0,0.175993
7,8,7.857998,CTR_REVIEW,CTR_FIX_OPPORTUNITY,17622,19.5,0.0,0.196274
8,9,7.567404,CTR_REVIEW,CTR_FIX_OPPORTUNITY,12275,14.7,0.0,0.196274
9,10,7.378072,CTR_REVIEW,CTR_FIX_OPPORTUNITY,7737,5.5,0.0,0.175993


In [50]:
output_path = "baseline_action_score.csv"

output_columns = [
    "rank",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue[output_columns].to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")

Saved: baseline_action_score.csv


In [51]:
display(
    queue[
        [
            "rank",
            "score",
            "action",
            "reason_code",
            "impressions_90d",
            "avg_position",
            "ctr",
            "ctr_position_rank"
        ]
    ].head(20)
)

,rank,score,action,reason_code,impressions_90d,avg_position,ctr,ctr_position_rank
0,1,10.092888,CTR_REVIEW,CTR_FIX_OPPORTUNITY,208678,9.7,0.0,0.175993
1,2,8.256017,CTR_REVIEW,CTR_FIX_OPPORTUNITY,22456,6.6,0.0,0.175993
2,3,8.053187,CTR_REVIEW,CTR_FIX_OPPORTUNITY,13676,4.3,0.0,0.154385
3,4,8.016233,CTR_REVIEW,CTR_FIX_OPPORTUNITY,16786,5.6,0.0,0.175993
4,5,7.984714,CTR_REVIEW,CTR_FIX_OPPORTUNITY,16156,9.0,0.0,0.175993
5,6,7.929072,CTR_REVIEW,CTR_FIX_OPPORTUNITY,15101,5.7,0.0,0.175993
6,7,7.896688,CTR_REVIEW,CTR_FIX_OPPORTUNITY,14519,7.4,0.0,0.175993
7,8,7.857998,CTR_REVIEW,CTR_FIX_OPPORTUNITY,17622,19.5,0.0,0.196274
8,9,7.567404,CTR_REVIEW,CTR_FIX_OPPORTUNITY,12275,14.7,0.0,0.196274
9,10,7.378072,CTR_REVIEW,CTR_FIX_OPPORTUNITY,7737,5.5,0.0,0.175993


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The baseline prioritizes pages with meaningful impressions and unusually low
CTR relative to pages in the same position bucket. The action is a review
recommendation, not proof that the page needs a CTR fix.

1. **Action:** CTR_REVIEW — **Why:** 208,678 impressions, position 9.7, and CTR 0.0 make this the largest visible CTR opportunity in the queue. **Confidence:** High for review priority. **Wrong if:** impressions or CTR tracking is incorrect, or the page has no real click opportunity for its query.

2. **Action:** CTR_REVIEW — **Why:** 22,456 impressions, position 6.6, and CTR 0.0 indicate strong visibility with no recorded clicks. **Confidence:** High. **Wrong if:** the measurement is incomplete or the query/page intent does not produce clicks.

3. **Action:** CTR_REVIEW — **Why:** 13,676 impressions, position 4.3, and CTR 0.0 indicate a visible page with no recorded clicks. **Confidence:** High. **Wrong if:** CTR data is missing or the page is not actually receiving relevant search impressions.

4. **Action:** CTR_REVIEW — **Why:** 16,786 impressions, position 5.6, and CTR 0.0 make this a high-visibility candidate. **Confidence:** High. **Wrong if:** the impression/click measurements are unreliable.

5. **Action:** CTR_REVIEW — **Why:** 16,156 impressions, position 9.0, and CTR 0.0 indicate poor recorded click response despite visibility. **Confidence:** High. **Wrong if:** the query intent does not normally lead to clicks.

6. **Action:** CTR_REVIEW — **Why:** 15,101 impressions, position 5.7, and CTR 0.0 suggest a potentially important CTR issue. **Confidence:** High. **Wrong if:** the zero CTR reflects tracking rather than user behavior.

7. **Action:** CTR_REVIEW — **Why:** 14,519 impressions, position 7.4, and CTR 0.0 indicate meaningful visibility without recorded clicks. **Confidence:** High. **Wrong if:** the underlying search data is incomplete.

8. **Action:** CTR_REVIEW — **Why:** 17,622 impressions, position 19.5, and CTR 0.0 make this a visible but lower-ranking CTR candidate. **Confidence:** Medium. **Wrong if:** the low position itself explains the lack of clicks.

9. **Action:** CTR_REVIEW — **Why:** 12,275 impressions, position 14.7, and CTR 0.0 indicate visibility with no recorded clicks. **Confidence:** Medium. **Wrong if:** position 14.7 is too low for a meaningful CTR-fix opportunity.

10. **Action:** CTR_REVIEW — **Why:** 7,737 impressions, position 5.5, and CTR 0.0 suggest a potentially fixable CTR problem. **Confidence:** High. **Wrong if:** clicks are under-recorded.

11. **Action:** CTR_REVIEW — **Why:** 7,732 impressions, position 8.3, and CTR 0.0 indicate reasonable visibility but no recorded clicks. **Confidence:** High. **Wrong if:** the page/query combination is not click-oriented.

12. **Action:** CTR_REVIEW — **Why:** 8,779 impressions, position 11.6, and CTR 0.0 indicate a possible CTR issue with moderate visibility. **Confidence:** Medium. **Wrong if:** the position is too weak to expect many clicks.

13. **Action:** CTR_REVIEW — **Why:** 6,635 impressions, position 5.2, and CTR 0.0 make this a visible candidate for review. **Confidence:** High. **Wrong if:** zero CTR is caused by data-quality problems.

14. **Action:** CTR_REVIEW — **Why:** 6,594 impressions, position 5.9, and CTR 0.0 indicate potential missed clicks. **Confidence:** High. **Wrong if:** the impression data does not represent relevant search exposure.

15. **Action:** CTR_REVIEW — **Why:** 6,527 impressions, position 9.4, and CTR 0.0 indicate moderate visibility with no recorded clicks. **Confidence:** Medium. **Wrong if:** the query has low click intent.

16. **Action:** CTR_REVIEW — **Why:** 6,524 impressions, position 9.3, and CTR 0.0 indicate a similar CTR opportunity. **Confidence:** Medium. **Wrong if:** tracking is incomplete or the result is not competitive for the query.

17. **Action:** CTR_REVIEW — **Why:** 6,402 impressions, position 8.1, and CTR 0.0 indicate visible search exposure without recorded clicks. **Confidence:** High. **Wrong if:** CTR is incorrectly recorded as zero.

18. **Action:** CTR_REVIEW — **Why:** 4,955 impressions, position 3.9, and CTR 0.0 are notable because the page ranks relatively well but records no clicks. **Confidence:** High. **Wrong if:** the query/page intent mismatch explains the absence of clicks.

19. **Action:** CTR_REVIEW — **Why:** 5,742 impressions, position 8.3, and CTR 0.0 indicate moderate visibility with no clicks. **Confidence:** Medium. **Wrong if:** impressions are not from relevant search queries.

20. **Action:** CTR_REVIEW — **Why:** 7,087 impressions, position 16.0, and CTR 0.0 indicate a visible but lower-ranking candidate. **Confidence:** Medium. **Wrong if:** the ranking position itself makes a CTR improvement unlikely to have much impact.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

### Weak picks

The weakest picks are the rows where the rule may confuse low CTR with a
true optimization opportunity. In particular, pages at positions around
11–20 may have low CTR simply because they rank too low to receive many
clicks. Zero CTR can also reflect tracking or measurement issues.

Therefore the baseline should be treated as a review queue, not as proof that
all selected pages require a CTR fix.

### Leakage check

The scoring rule uses only observable fields available in the current
snapshot:

- impressions_90d
- avg_position
- ctr
- position bucket / within-position CTR rank

It does not use:

- trend_direction
- trend_pct
- is_declining_label
- future performance
- any future-window label

`trend_direction` was used only to audit signals, not to calculate the final
score.

In [52]:
score_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "position_bucket",
    "ctr_position_rank"
]

print("Features used by baseline:")
for col in score_features:
    print("-", col)

forbidden = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("\nLeakage check:")
for col in forbidden:
    print(f"{col}: {'FOUND IN SCORE' if col in score_features else 'NOT USED'}")

Features used by baseline:
- impressions_90d
- avg_position
- ctr
- position_bucket
- ctr_position_rank

Leakage check:
trend_direction: NOT USED
trend_pct: NOT USED
is_declining_label: NOT USED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.